In [ ]:
import json, glob, csv, re, pandas as pd

TOOLS = ["kics", "checkov", "trivy", "azpolicy_whatif", "azpolicy_cont", "kql"]
MCS = [f"mc{i:02d}" for i in range(1, 11)]

MIN_RUNS = {"kics": 2, "checkov": 2, "trivy": 2}   # everything else: 1

rows, manual = [], []


# --- static tiers: results/normalised/*.json -------------------------------
for f in glob.glob("../results/normalised/*.json"):
    r = json.load(open(f))
    for fd in r["findings"]:
        # JSON stores the full flag name ("mc01_public_storage"); the grid uses
        # the short id. Without this split the reindex below matches nothing.
        rows.append({"tool": r["tool"], "variant": r["variant"],
                     "run": r["run"], "mc": fd["mc_id"].split("_")[0]})

# --- deployed-state: mc_detected is a ';'-separated list of full flag names -
for r in csv.DictReader(open("../results/policy_scan_log.csv")):
    for m in filter(None, (r["mc_detected"] or "").split(";")):
        rows.append({"tool": "azpolicy_cont", "variant": r["variant"],
                     "run": int(r["run"]), "mc": m.split("_")[0]})

# --- what-if: a policy denial is the detection signal ----------------------
for r in csv.DictReader(open("../results/policy_whatif_log.csv")):
    if r["blocked"].strip().lower() == "true" and r["denied_by"]:
        m = re.match(r"(mc[0-9]+)", r["denied_by"])   # 'mc01-deny-public-storage'
        if m:
            rows.append({"tool": "azpolicy_whatif", "variant": r["variant"],
                         "run": int(r["run"]), "mc": m.group(1)})

# --- runtime KQL: headerless variant,run,tool,mc,status,ttf ----------------
# MANUAL_CHECK_REQUIRED is neither a detection nor a miss -- it means the rule
# was never actually run (mc05/mc10b are outside ARG's reach; mc07 is suppressed
# unless RUNTIME_CHECK_MC07=1). Collected separately so it can't be silently
# scored as a clean negative.
for r in csv.reader(open("../results/run_log_runtime.csv")):
    if len(r) < 6:
        continue
    v, run, tool, mc, status, ttf = r[:6]
    if status == "MANUAL_CHECK_REQUIRED":
        manual.append({"tool": "kql", "variant": v, "mc": mc})
    elif status != "NOT_DETECTED":
        rows.append({"tool": "kql", "variant": v, "run": int(run), "mc": mc})

df = pd.DataFrame(rows)
manual_df = pd.DataFrame(manual)

# a variant detects mc if the tool flags it in >= MIN_RUNS of its runs
vuln = df[df.variant.str.startswith("vuln")]
hits = (vuln.groupby(["tool", "mc"]).run.nunique()
            .reset_index(name="runs_hit"))
hits["detected"] = hits.apply(
    lambda r: int(r.runs_hit >= MIN_RUNS.get(r.tool, 1)), axis=1)

grid = (hits.pivot_table(index="mc", columns="tool",
                         values="detected", fill_value=0)
            .reindex(index=MCS, columns=TOOLS, fill_value=0).astype(int))

# kql/mc07 must NOT be scored as a detection. run_log_runtime.csv holds two mc07
# hits (vuln-07, mixed-202) produced by deliberately forcing RUNTIME_CHECK_MC07=1,
# but 02-results.md section 7 establishes these are coincidental agreement from a
# query that fires unconditionally: Resource Graph does not reliably index
# microsoft.insights/diagnosticsettings, so the isempty(diag) join is
# structurally always true regardless of ground truth. Counting it would
# (a) overstate kql as 9/10 rather than 8/10, and (b) make kql a spurious
# superset of azpolicy_cont -- erasing the section 5 headline that mc07 is
# caught by the deployed-state tier and nowhere else.
DISPUTED = [("kql", "mc07")]
for _tool, _mc in DISPUTED:
    grid.loc[_mc, _tool] = 0

grid.to_csv("../results/coverage_grid.csv")
print(grid)
print("\ncoverage per tool:")
print(grid.sum().to_string())
print("\nflags detected by NO tool:", [m for m in MCS if grid.loc[m].sum() == 0])
print("excluded as structurally unreliable:", DISPUTED)
if len(manual_df):
    print("\nkql MANUAL_CHECK_REQUIRED (not scored either way):",
          sorted(manual_df[manual_df.variant.str.startswith('vuln')].mc.unique()))


In [12]:
import math

def wilson_ci(k, n, z=1.96):
    """95% Wilson score interval for k successes in n trials."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    denom = 1 + z*z/n
    centre = (p + z*z/(2*n)) / denom
    half = (z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))) / denom
    # clamp to [0, 1]. At k=0 the true lower bound is exactly 0, but floating
    # point makes centre-half ~= -1.4e-17, which round() renders as "-0.0" --
    # a negative probability in a results table reads as a real error.
    lo = max(0.0, centre - half)
    hi = min(1.0, centre + half)
    return (round(lo, 3), round(hi, 3))

N = 10
tpr = []
for tool in grid.columns:
    tp = int(grid[tool].sum())
    lo, hi = wilson_ci(tp, N)
    tpr.append({"tool": tool, "tp": tp, "tpr": round(tp/N, 3),
                "ci95_low": lo, "ci95_high": hi})
tpr = pd.DataFrame(tpr)
print(tpr)

              tool  tp  tpr  ci95_low  ci95_high
0             kics   3  0.3     0.108      0.603
1          checkov   2  0.2     0.057      0.510
2            trivy   5  0.5     0.237      0.763
3  azpolicy_whatif   4  0.4     0.168      0.687
4    azpolicy_cont   5  0.5     0.237      0.763
5              kql   9  0.9     0.596      0.982


In [13]:
compliant = df[df.variant.isin(["hardened", "noisy-compliant"])]
OPP = 2 * 10   # 2 compliant variants x 10 fault-topics = scope-relevant opportunities

fpr = []
for tool in grid.columns:
    # scope-relevant FPs = distinct mapped mc0X flagged on a compliant variant,
    # deduped per variant (a tool flagging the same mc across 3 runs counts once)
    sr_h = compliant[(compliant.tool==tool) & (compliant.variant=="hardened")].mc.nunique()
    sr_n = compliant[(compliant.tool==tool) & (compliant.variant=="noisy-compliant")].mc.nunique()
    fp = sr_h + sr_n
    lo, hi = wilson_ci(fp, OPP)
    fpr.append({"tool": tool, "fp_hardened": sr_h, "fp_noisy": sr_n,
                "fpr": round(fp/OPP, 3), "ci95_low": lo, "ci95_high": hi})
fpr = pd.DataFrame(fpr)
print(fpr)

              tool  fp_hardened  fp_noisy  fpr  ci95_low  ci95_high
0             kics            0         0  0.0       0.0      0.161
1          checkov            0         0  0.0       0.0      0.161
2            trivy            0         0  0.0       0.0      0.161
3  azpolicy_whatif            0         0  0.0       0.0      0.161
4    azpolicy_cont            0         0  0.0       0.0      0.161
5              kql            0         0  0.0       0.0      0.161


In [ ]:
import numpy as np

rt = pd.read_csv("../results/run_log_runtime.csv", header=None,
                 names=["variant", "run", "tool", "mc",
                        "detected_at_utc", "ttf_seconds"])

# Exclude BOTH non-detections. Filtering on != "NOT_DETECTED" alone leaves the
# MANUAL_CHECK_REQUIRED rows in, whose ttf_seconds is "-" -> NaN. That does not
# crash: .median() skips NaN so it still prints a plausible number, but the
# bootstrap takes np.median over samples containing NaN and the int cast turns
# the CI into -9223372036854775808 (INT64_MIN), while n is overstated (33 vs 20).
NON_DETECTIONS = ["NOT_DETECTED", "MANUAL_CHECK_REQUIRED"]
det = rt[~rt.detected_at_utc.isin(NON_DETECTIONS)].copy()
det["ttf"] = pd.to_numeric(det["ttf_seconds"], errors="coerce")

def boot_median_ci(x, iters=10000, seed=42):
    """Percentile bootstrap 95% CI for the median."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    meds = [np.median(rng.choice(x, size=len(x), replace=True)) for _ in range(iters)]
    return float(np.median(x)), tuple(np.percentile(meds, [2.5, 97.5]))

ttf = det.ttf.dropna()
median, (lo, hi) = boot_median_ci(ttf)

# counts of the two non-detection kinds, kept distinct: NOT_DETECTED means the
# rule ran and found nothing; MANUAL_CHECK_REQUIRED means it never ran at all
# (mc05/mc10b are outside ARG's reach, mc07 is suppressed unless
# RUNTIME_CHECK_MC07=1), so it is not evidence of a clean negative.
nd = int((rt.detected_at_utc == "NOT_DETECTED").sum())
mn = int((rt.detected_at_utc == "MANUAL_CHECK_REQUIRED").sum())

print(f"KQL time-to-flag (cold deploy -> ARG detection), n={len(ttf)}")
print(f"  n={len(ttf)}  median={median:.1f}s  "
      f"range=[{ttf.min():.0f},{ttf.max():.0f}]  CI95=[{lo:.0f}, {hi:.0f}]")
print(f"  median {median:.1f}s   95% bootstrap CI [{lo:.1f}, {hi:.1f}]")
print(f"  min {ttf.min():.0f}s   max {ttf.max():.0f}s   NOT_DETECTED={nd}   MANUAL_CHECK_REQUIRED={mn}")
print(f"  values: {sorted(ttf.astype(int))}")

# Contrast for the write-up: azpolicy_cont's forced compliance re-scan runs
# 459-826s (results/policy_scan_log.csv). Separate labelled axes; never average.


In [ ]:
from itertools import combinations
from scipy.stats import binomtest

def holm(pvals):
    """Holm-Bonferroni step-down adjusted p-values, monotonicity enforced.
    15 pairwise tests uncorrected would expect ~0.75 false positives at a=0.05."""
    m = len(pvals)
    order = sorted(range(m), key=lambda i: pvals[i])
    adj, running = [0.0] * m, 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * pvals[i])
        adj[i] = min(1.0, running)
    return adj

def cohen_g(a_only, b_only):
    disc = a_only + b_only
    return (0.0, 0) if disc == 0 else ((a_only / disc) - 0.5, disc)

def band(g, disc, floor=6):
    """Cohen's g band, suppressed where disc is too small to mean anything.

    g = (a/disc) - 0.5 saturates at +-0.5 whenever ALL discordant pairs fall one
    way -- whether that is 1 flag or 100. Reporting 'large' off disc=1 measures
    unanimity, not magnitude. floor=6 is the exact-McNemar power floor: the
    smallest achievable two-sided p is 2*0.5**disc, so below disc=6 no result
    under 0.05 is reachable regardless of the data.
    """
    if disc == 0:
        return "no discordance"
    if disc < floor:
        return f"uninterpretable (disc<{floor})"
    ag = abs(g)
    return ("negligible" if ag < 0.05 else "small" if ag < 0.15
            else "medium" if ag < 0.25 else "large")

pairs = []
for At, Bt in combinations(grid.columns, 2):
    a = int(((grid[At] == 1) & (grid[Bt] == 0)).sum())   # A caught, B missed
    b = int(((grid[At] == 0) & (grid[Bt] == 1)).sum())   # B caught, A missed
    g, disc = cohen_g(a, b)
    p = binomtest(min(a, b), disc, 0.5).pvalue if disc else 1.0
    pmin = binomtest(0, disc, 0.5).pvalue if disc else 1.0   # best case for this disc
    pairs.append(dict(A=At, B=Bt, a=a, b=b, disc=disc, g=g, p=p, pmin=pmin))

for pr, adj in zip(pairs, holm([x["p"] for x in pairs])):
    pr["p_holm"] = adj

hdr = (f"{'A':16}{'B':16}{'a':>2}{'b':>3}{'disc':>5}{'g':>8}  "
       f"{'band':<26}{'p':>7}{'p_holm':>8}{'min_p':>7}  powered?")
print(hdr); print("-" * len(hdr))
for x in pairs:
    print(f"{x['A']:16}{x['B']:16}{x['a']:>2}{x['b']:>3}{x['disc']:>5}{x['g']:>+8.3f}  "
          f"{band(x['g'], x['disc']):<26}{x['p']:>7.3f}{x['p_holm']:>8.3f}"
          f"{x['pmin']:>7.3f}  {'yes' if x['pmin'] < 0.05 else 'NO'}")

sig = [x for x in pairs if x["p_holm"] < 0.05]
unp = [x for x in pairs if x["pmin"] >= 0.05]
print(f"\nSignificant after Holm (alpha=0.05): {len(sig)} of {len(pairs)}")
print(f"Structurally unable to reach p<0.05 at n=10: {len(unp)} of {len(pairs)}")

# The defensible primary result: a deterministic set relation over 10 flags,
# which needs no significance test at all.
print("\nStrict dominance (superset relation, no test required):")
for x in pairs:
    if x["a"] == 0 and x["b"] > 0:
        print(f"  {x['B']:15} strictly dominates {x['A']:15} (+{x['b']} flags, 0 missed)")
    elif x["b"] == 0 and x["a"] > 0:
        print(f"  {x['A']:15} strictly dominates {x['B']:15} (+{x['a']} flags, 0 missed)")


In [ ]:
from itertools import combinations

S = {t: set(grid.index[grid[t] == 1]) for t in grid.columns}
ALL = set(grid.index)

print("single-tool coverage:")
for t, s in sorted(S.items(), key=lambda x: -len(x[1])):
    print(f"  {t:16} {len(s)}/10   misses {sorted(ALL - s)}")

print("\nbest union at each size (joint FPR is 0 for every tool -- see fpr table,")
print("so coverage is the only discriminator here):")
for n in (1, 2, 3):
    best = sorted(((len(set().union(*[S[t] for t in c])), c)
                   for c in combinations(S, n)), reverse=True)[0]
    cnt, combo = best
    print(f"  {n}-tool: {' + '.join(combo):45} = {cnt}/10  misses {sorted(ALL - set().union(*[S[t] for t in combo]))}")

print("\nmarginal gain of adding each tool to the best single tool (kql):")
for t in S:
    if t != "kql":
        print(f"  kql + {t:16} adds {len(S[t] - S['kql'])} flag(s): {sorted(S[t] - S['kql']) or '-'}")

print("\nbest combination EXCLUDING kql (discounts the rule-authorship advantage:")
print("the kql rules were written against the same CIS controls the faults violate):")
noq = {k: v for k, v in S.items() if k != "kql"}
for n in (1, 2, 3):
    cnt, combo = sorted(((len(set().union(*[noq[t] for t in c])), c)
                         for c in combinations(noq, n)), reverse=True)[0]
    print(f"  {n}-tool: {' + '.join(combo):45} = {cnt}/10  misses {sorted(ALL - set().union(*[noq[t] for t in combo]))}")

print(f"\nreachable by NO combination of all {len(S)} tools: {sorted(ALL - set().union(*S.values()))}")


In [ ]:
import os
import matplotlib.pyplot as plt, seaborn as sns

# matplotlib will not create the output directory itself -- savefig raises
# FileNotFoundError if it is missing.
os.makedirs("figures", exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(grid, annot=True, cbar=False, cmap="Blues",
            linewidths=.5, ax=ax)

# Title derived from grid.shape rather than hardcoded: the column count changed
# from 7 to 6 when 'defender' was dropped, and a hardcoded caption silently went
# stale. Deriving it means the figure can never disagree with the data again.
n_mc, n_tools = grid.shape
ax.set_title(f"Detection coverage: {n_tools} tools x {n_mc} misconfigurations")
ax.set_xlabel(""); ax.set_ylabel("")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

fig.tight_layout()
fig.savefig("figures/coverage_heatmap.png", dpi=200)
print(f"wrote figures/coverage_heatmap.png  ({n_tools} tools x {n_mc} flags)")
